In [103]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# **Cleaning team_match data**

In [104]:
team_matches = pd.read_csv("/content/team_matches_home_away_raw.csv.csv")

In [105]:
print("Players Information:", team_matches.shape)

Players Information: (15808, 19)


In [106]:
team_matches.head()

,id,team_name,round,match_date,year,home_away,opponent,team_quarter_scores,team_score,opponent_quarter_scores,opponent_score,result,margin,venue,crowd,team_goals_kicked,team_behinds,opponent_goals_kicked,opponent_behinds
0,15807,Hawthorn Hawks,QF,1994-09-10,1994,A,North Melbourne Kangaroos,4.5 5.7 10.11 13.13 13.13 13.13,91,2.3 6.12 9.12 12.19 13.23 15.24,114,L,23,Waverley Park,38223.0,13,13,15,24
1,15808,North Melbourne Kangaroos,QF,1994-09-10,1994,H,Hawthorn Hawks,2.3 6.12 9.12 12.19 13.23 15.24,114,4.5 5.7 10.11 13.13 13.13 13.13,91,W,6,Waverley Park,38223.0,15,24,13,13
2,5646,North Melbourne Kangaroos,10,2008-05-31,2008,A,Brisbane Lions,2.2 6.2 12.3 15.8,98,4.7 11.12 15.17 18.21,129,L,-31,The Gabba,22118.0,15,8,18,21
3,8829,Sydney Swans,15,2017-06-30,2017,A,Melbourne Demons,1.8 5.15 8.16 11.19,85,4.0 4.1 5.4 7.8,50,W,35,Melbourne Cricket Ground,47464.0,11,19,7,8
4,8873,Sydney Swans,11,2019-06-01,2019,A,Geelong Cats,3.3 5.8 6.12 8.15,63,5.1 7.2 11.4 13.7,85,L,-22,GMHBA Stadium,29021.0,8,15,13,7


In [107]:
team_matches.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15808 entries, 0 to 15807
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       15808 non-null  int64  
 1   team_name                15808 non-null  object 
 2   round                    15808 non-null  object 
 3   match_date               15808 non-null  object 
 4   year                     15808 non-null  int64  
 5   home_away                15808 non-null  object 
 6   opponent                 15808 non-null  object 
 7   team_quarter_scores      15808 non-null  object 
 8   team_score               15808 non-null  int64  
 9   opponent_quarter_scores  15808 non-null  object 
 10  opponent_score           15808 non-null  int64  
 11  result                   15808 non-null  object 
 12  margin                   15808 non-null  int64  
 13  venue                    15808 non-null  object 
 14  crowd                 

In [108]:
team_matches.isnull().sum()

,0
id,0
team_name,0
round,0
match_date,0
year,0
home_away,0
opponent,0
team_quarter_scores,0
team_score,0
opponent_quarter_scores,0


In [109]:
print("Datatypes before cleaning",team_matches.dtypes)

Datatypes before cleaning id                           int64
team_name                   object
round                       object
match_date                  object
year                         int64
home_away                   object
opponent                    object
team_quarter_scores         object
team_score                   int64
opponent_quarter_scores     object
opponent_score               int64
result                      object
margin                       int64
venue                       object
crowd                      float64
team_goals_kicked            int64
team_behinds                 int64
opponent_goals_kicked        int64
opponent_behinds             int64
dtype: object


In [110]:
# Fix date data type
team_matches["match_date"] = pd.to_datetime(team_matches["match_date"],errors="coerce")

# Fix crowd data type
team_matches["crowd"] = pd.to_numeric(team_matches["crowd"],errors="coerce")

In [111]:
team_matches.drop_duplicates()
# Calculate median
median_crowd = team_matches["crowd"].median()
print("\nMedian Crowd:", median_crowd)
# Fill missing values with median
team_matches["crowd"] = team_matches["crowd"].fillna(median_crowd)
team_matches["crowd"] = team_matches["crowd"].astype("Int64")


Median Crowd: 29814.0


In [112]:
print(team_matches.isnull().sum())

id                         0
team_name                  0
round                      0
match_date                 0
year                       0
home_away                  0
opponent                   0
team_quarter_scores        0
team_score                 0
opponent_quarter_scores    0
opponent_score             0
result                     0
margin                     0
venue                      0
crowd                      0
team_goals_kicked          0
team_behinds               0
opponent_goals_kicked      0
opponent_behinds           0
dtype: int64


In [113]:
print("\nDatatypes After Cleaning:")
print(team_matches.dtypes)


Datatypes After Cleaning:
id                                  int64
team_name                          object
round                              object
match_date                 datetime64[ns]
year                                int64
home_away                          object
opponent                           object
team_quarter_scores                object
team_score                          int64
opponent_quarter_scores            object
opponent_score                      int64
result                             object
margin                              int64
venue                              object
crowd                               Int64
team_goals_kicked                   int64
team_behinds                        int64
opponent_goals_kicked               int64
opponent_behinds                    int64
dtype: object


In [114]:
team_matches.to_csv("Team_Matches_Cleaned.csv", index=False)
print("Cleaning completed!")

Cleaning completed!


# **Task Implementation**

In [126]:
player_stat = pd.read_csv("/content/Players_round_by_round_stat_cleaned.csv")
matches = pd.read_csv("/content/Team_Matches_Cleaned.csv")

print("Players dataset:", player_stat.shape)
print("Matches dataset:", matches.shape)

Players dataset: (274079, 35)
Matches dataset: (15808, 19)


In [127]:
common_columns = player_stat.columns.intersection(matches.columns)
print("Common columns:")
print(list(common_columns))

# Make team names consistent
player_stat["team"] = player_stat["team"].str.strip()
matches["team_name"] = matches["team_name"].str.strip()
matches["team_name"] = matches["team_name"].replace("W. Bulldogs", "Western Bulldogs")

# Rename team_name so both datasets have the same column name
matches = matches.rename(columns={"team_name": "team"})

# Check whether the possible merge key is unique in match data
merge_key = ["team", "round", "year", "match_date"]
print("\nMerge key:")
print(merge_key)

Common columns:
['id', 'year', 'opponent', 'round', 'result', 'match_date', 'margin']

Merge key:
['team', 'round', 'year', 'match_date']


In [128]:
duplicate_keys = matches.duplicated(merge_key).sum()
print("Duplicate merge keys in Team Match dataset:")
print(duplicate_keys)

Duplicate merge keys in Team Match dataset:
0


# **Context Enrichment**

In [129]:
player_stat["team"] = player_stat["team"].astype(str)
matches["team"] = matches["team"].astype(str)

player_stat["round"] = player_stat["round"].astype(str)
matches["round"] = matches["round"].astype(str)

player_stat["year"] = player_stat["year"].astype(str)
matches["year"] = matches["year"].astype(str)

player_stat["match_date"] = player_stat["match_date"].astype(str)
matches["match_date"] = matches["match_date"].astype(str)

In [130]:
match_context = matches[merge_key + ["home_away", "venue", "crowd"]]

print(match_context.head())

                        team round  year  match_date home_away  \
0             Hawthorn Hawks    QF  1994  1994-09-10         A   
1  North Melbourne Kangaroos    QF  1994  1994-09-10         H   
2  North Melbourne Kangaroos    10  2008  2008-05-31         A   
3               Sydney Swans    15  2017  2017-06-30         A   
4               Sydney Swans    11  2019  2019-06-01         A   

                      venue  crowd  
0             Waverley Park  38223  
1             Waverley Park  38223  
2                 The Gabba  22118  
3  Melbourne Cricket Ground  47464  
4             GMHBA Stadium  29021  


In [131]:
# Merge player data with match context
enriched_players = player_stat.merge(match_context,on=merge_key,how="left")

In [132]:
print(enriched_players.head())

       id              team  year  career_game_count          opponent round  \
0  556392    Hawthorn Hawks  1994                 17   Richmond Tigers    21   
1  614897      Geelong Cats  2024                  1   St Kilda Saints     1   
2  583553  Essendon Bombers  1999                 97    Adelaide Crows    10   
3  590676  Western Bulldogs  1994                 36   St Kilda Saints    21   
4  582473   Richmond Tigers  1997                113  Melbourne Demons    10   

  result  jersey_num  kicks  marks  ...  bounces  goal_assist  \
0      W          34      5      4  ...        0            0   
1      W           7      5      0  ...        0            0   
2      W           6     14      5  ...        0            0   
3      W          35     12     10  ...        0            0   
4      L          41      4      2  ...        0            0   

   percentage_of_game_played  player_id  match_date  fantasy_points  margin  \
0                        0.0      45552  1994-08-

In [133]:
print("New columns:")
print(enriched_players[["home_away", "venue", "crowd"]].head())

New columns:
  home_away                     venue  crowd
0         A  Melbourne Cricket Ground  52562
1         H             GMHBA Stadium  39352
2         A              AAMI Stadium  39389
3         A         Waverley Park\r\n  14653
4         H  Melbourne Cricket Ground  28879


In [134]:
print("Original player records:")
print(len(player_stat))

print("\nRecords after merge:")
print(len(enriched_players))

Original player records:
274079

Records after merge:
274079


In [135]:
enriched_players.to_csv("Players_Round_By_Round_Enriched.csv",index=False)
print("Enriched dataset saved successfully.")

Enriched dataset saved successfully.


# **Merge Validation**

In [136]:
print("Missing home/away:", enriched_players["home_away"].isna().sum())
print("Missing venue:", enriched_players["venue"].isna().sum())
print("Missing crowd:", enriched_players["crowd"].isna().sum())

Missing home/away: 0
Missing venue: 0
Missing crowd: 0


In [137]:
print("Duplicate records:", enriched_players.duplicated().sum())

Duplicate records: 0


In [138]:
print("Before merge:", len(player_stat))
print("After merge:", len(enriched_players))

Before merge: 274079
After merge: 274079


# **Contextual Analysis**

In [139]:
data = pd.read_csv("/content/Players_Round_By_Round_Enriched.csv")

In [140]:
home_away = enriched_players.groupby("home_away")["fantasy_points"].mean()
print(home_away)
print("Home average:", round(home_away["H"], 2))
print("Away average:", round(home_away["A"], 2))

home_away
A    64.004408
H    66.485504
Name: fantasy_points, dtype: float64
Home average: 66.49
Away average: 64.0


In [141]:
print(enriched_players[["crowd", "fantasy_points"]].corr())

                   crowd  fantasy_points
crowd           1.000000        0.015063
fantasy_points  0.015063        1.000000


In [142]:
venue_result = enriched_players.groupby("venue")["fantasy_points"].mean()
print(venue_result.sort_values(ascending=False).head(10))
print("Highest venue:", venue_result.idxmax())
print("Average fantasy points:", round(venue_result.max(), 2))

venue
Junction Oval        75.333333
TIO Stadium\r\n      74.187500
Jiangwan Stadium     72.481752
UTAS Stadium\r\n     70.879699
Riverway Stadium     70.369565
Westpac Stadium      69.716418
TIO Traeger Park     69.615385
Accor Stadium\r\n    68.866667
Mars Stadium         68.040689
UTAS Stadium         67.962007
Name: fantasy_points, dtype: float64
Highest venue: Junction Oval
Average fantasy points: 75.33


In [144]:
merged = player_stat.merge(match_context,on=merge_key,how="left")
print("Duplicate merge keys:", duplicate_keys)

print("Original records:", len(player_stat))
print("Merged records:", len(merged))

print("Duplicate records:", merged.duplicated().sum())

print("Missing home/away:", merged["home_away"].isna().sum())
print("Missing venue:", merged["venue"].isna().sum())
print("Missing crowd:", merged["crowd"].isna().sum())

Duplicate merge keys: 0
Original records: 274079
Merged records: 274079
Duplicate records: 0
Missing home/away: 0
Missing venue: 0
Missing crowd: 0


In [146]:
print("DATA QUALITY REPORT")
print()

print("Merge Key:")
print("team + round + year + match_date")

print("\nMerge Strategy:")
print("Left merge")

print("\nDuplicate Match Keys:")
print(duplicate_keys)

print("\nOriginal Player Records:")
print(len(player_stat))

print("\nRecords After Merge:")
print(len(merged))

print("\nDuplicate Records After Merge:")
print(merged.duplicated().sum())

print("\nMissing Home/Away:")
print(merged["home_away"].isna().sum())

print("\nMissing Venue:")
print(merged["venue"].isna().sum())

print("\nMissing Crowd:")
print(merged["crowd"].isna().sum())

print("\nData Quality Issue:")
print("W. Bulldogs and Western Bulldogs were standardized.")

print("\nAssumption:")
print("team + round + year + match_date identifies the correct match.")

DATA QUALITY REPORT

Merge Key:
team + round + year + match_date

Merge Strategy:
Left merge

Duplicate Match Keys:
0

Original Player Records:
274079

Records After Merge:
274079

Duplicate Records After Merge:
0

Missing Home/Away:
0

Missing Venue:
0

Missing Crowd:
0

Data Quality Issue:
W. Bulldogs and Western Bulldogs were standardized.

Assumption:
team + round + year + match_date identifies the correct match.
